In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
# walks upward until it finds a directory containing ppm/
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents) if (p / "ppm").is_dir()),
    current_dir,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from ppm.wandb_utils import load_multiple_experiments

plots_dir = PROJECT_ROOT / "plots/features"
plots_dir.mkdir(parents=True, exist_ok=True)

In [8]:
PROJECTS=[
  #"BPI12_spark_freeze_002",
  #"BPI17_spark_freeze_001",
  "bpi12_spark_gpt2-mini_finetune_complete_001",
  "bpi12_spark_gpt2_finetune_complete_001 ",
  "bpi17_spark_gpt2-mini_finetune_complete_001",
  "bpi17_spark_gpt2-finetune_complete_001",
  "BPI12_spark_freeze_001",
  "BBPI12_spark_freeze_001",
  "activity_only_bpi12_002",
  "extra_features_bpi12_001",
  "extra_features_bpi12_003",
  "activity_only_unfreeze_last_bpi12_002",
  "extra_feature_bpi12_unfreeze_last_001",
  "gpt2_unfreeze_last_002",
  "extra_feature_bpi12_unfreeze_last2_001",
  "extra_features_bpi12_unfreeze_last2_001",
  "gpt2_unfreeze_last2_001",
  "extra_features_bpi12_unfreeze_last3_001",
  "extra_features_bpi12_unfreeze_last4_001",
]

runs_raw, _ = load_multiple_experiments(PROJECTS, force_update=False)


Database already exists: /app/visualization/paper/metrics/bpi12_spark_gpt2-mini_finetune_complete_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/bpi12_spark_gpt2_finetune_complete_001 .db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/bpi17_spark_gpt2-mini_finetune_complete_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/bpi17_spark_gpt2-finetune_complete_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI12_spark_freeze_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BBPI12_spark_freeze_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/activity_only_bpi12_002.db
Use force_update=True to re-fetch from wandb
Database already exi

In [9]:
import pandas as pd

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don't wrap/truncate by width
pd.set_option("display.max_colwidth", None)  # optional: full cell content

runs_raw.columns

Index(['id', 'name', 'r', 'lr', 'log', 'device', 'epochs', 'compile',
       'n_heads', 'backbone',
       ...
       'group', 'job_type', 'project', 'best_test_final_next_activity_auroc',
       'test_next_activity_auroc', 'train_next_activity_auroc',
       'val_next_activity_auroc', 'n_embd', 'n_head', 'block_size'],
      dtype='str', length=101)

In [10]:
# prepocessing
runs = runs_raw.copy()

fine_tuning_col = "fine_tuning" if "fine_tuning" in runs.columns else "fine-tuning"
freeze_layers_col = "freeze_layers" if "freeze_layers" in runs.columns else "freeze-layers"
mask_no_finetuning = runs[fine_tuning_col].isna() | (runs[fine_tuning_col].astype(str) == "None")
runs.loc[mask_no_finetuning, freeze_layers_col] = "all"


## Impact freezing

In [11]:
METRIC = "best_test_final_next_activity_acc"
#METRIC = "best_test_final_next_activity_f1"

for col in ["categorical_features", "continuous_features"]:
    if col in runs.columns:
        runs[col] = runs[col].astype(str)

GROUP_COLS = ["log", "backbone", "freeze_layers", "lr", "batch_size", "categorical_features", "continuous_features"]

EXTRA_COLS = ["project", "total_params", "trainable_params", "embedding_size"]
EXTRA_COLS = [c for c in EXTRA_COLS if c in runs.columns]  # only keep ones that exist

# Build df with everything we need
df = runs[GROUP_COLS + ["id", METRIC] + EXTRA_COLS].dropna(subset=[METRIC]).copy()
df[METRIC] = df[METRIC].astype(float)

# 1) Metric stats (ONLY from METRIC)
agg = (
    df.groupby(GROUP_COLS)[METRIC]
      .agg(["count", "mean", "std", "min", "max"])
      .rename(columns={
          "count": "n_runs",
          "mean": "acc_mean",
          "std": "acc_std",
          "min": "acc_min",
          "max": "acc_max",
      })
)

# 2) Add “descriptor” columns (not metric-related)
#    We assume these are constant within group; if not, we’ll detect it below.
descriptors = df.groupby(GROUP_COLS)[EXTRA_COLS].first() if EXTRA_COLS else None

# Optional: detect groups where descriptors are NOT constant
if EXTRA_COLS:
    nunique = df.groupby(GROUP_COLS)[EXTRA_COLS].nunique(dropna=False)
    bad = (nunique > 1).any(axis=1)
    if bad.any():
        print("Warning: some groups have varying descriptor values:")
        display(nunique[bad].reset_index())

# 3) Best run id per group (based on METRIC)
best_idx = df.groupby(GROUP_COLS)[METRIC].idxmax()
best_runs = df.loc[best_idx].set_index(GROUP_COLS)["id"].rename("best_run_id")

# 4) Combine
summary = agg
if descriptors is not None:
    summary = summary.join(descriptors)

summary = summary.join(best_runs).reset_index()
summary["n_runs"] = summary["n_runs"].astype(int)

for col in ["acc_mean", "acc_std", "acc_min", "acc_max"]:
    summary[col] = summary[col].map(lambda x: f"{x:.4f}")

summary = summary.sort_values(GROUP_COLS).reset_index(drop=True)

with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 200):
    display(summary)


,log,backbone,freeze_layers,lr,batch_size,categorical_features,continuous_features,project,total_params,trainable_params,embedding_size
0,BPI12,gpt2,-1,0.005,64.0,['activity'],['accumulated_time'],2,1,1,1
1,BPI12,gpt2,"-1,-2",0.005,64.0,"['activity', 'resource']","['accumulated_time', 'amount']",2,1,1,1
2,BPI12,gpt2,"-1,-2",0.005,64.0,"['activity', 'resource']",['accumulated_time'],2,1,1,1
3,BPI12,gpt2,"-1,-2",0.005,64.0,['activity'],"['accumulated_time', 'amount']",2,1,1,1
4,BPI12,gpt2,"-1,-2",0.005,64.0,['activity'],['accumulated_time'],3,1,1,1
5,BPI12,gpt2,"1,-1",0.005,64.0,"['activity', 'resource']","['accumulated_time', 'amount']",2,1,1,1


,log,backbone,freeze_layers,lr,batch_size,categorical_features,continuous_features,n_runs,acc_mean,acc_std,acc_min,acc_max,project,total_params,trainable_params,embedding_size,best_run_id
0,BPI12,gpt2,-1,0.00500,64.0,"['activity', 'resource']","['accumulated_time', 'amount']",10,0.8354,0.0051,0.8275,0.8433,extra_feature_bpi12_unfreeze_last_001,124534299.0,7182363.0,768.0,p9ixmkgb
1,BPI12,gpt2,-1,0.00500,64.0,"['activity', 'resource']",['accumulated_time'],10,0.8290,0.0089,0.8143,0.8439,extra_feature_bpi12_unfreeze_last_001,124533531.0,7181595.0,768.0,olsrjn2u
2,BPI12,gpt2,-1,0.00500,64.0,['activity'],"['accumulated_time', 'amount']",10,0.8022,0.0153,0.7743,0.8155,extra_feature_bpi12_unfreeze_last_001,124486683.0,7134747.0,768.0,c8k6qu2a
3,BPI12,gpt2,-1,0.00500,64.0,['activity'],['accumulated_time'],20,0.8038,0.0089,0.7788,0.8165,extra_feature_bpi12_unfreeze_last_001,124485915.0,7133979.0,768.0,ww93fj5c
4,BPI12,gpt2,-1,0.00500,64.0,['activity'],[],10,0.7860,0.0146,0.7622,0.8033,activity_only_unfreeze_last_bpi12_002,124484379.0,7132443.0,768.0,84j30zbq
5,BPI12,gpt2,"-1,-2",0.00500,64.0,"['activity', 'resource']","['accumulated_time', 'amount']",15,0.8342,0.0089,0.8176,0.8495,extra_feature_bpi12_unfreeze_last2_001,124534299.0,14270235.0,768.0,r9p8v5nq
6,BPI12,gpt2,"-1,-2",0.00500,64.0,"['activity', 'resource']",['accumulated_time'],15,0.8314,0.0056,0.8206,0.8399,extra_feature_bpi12_unfreeze_last2_001,124533531.0,14269467.0,768.0,tcvhyed1
7,BPI12,gpt2,"-1,-2",0.00500,64.0,['activity'],"['accumulated_time', 'amount']",15,0.8093,0.0087,0.7915,0.8228,extra_feature_bpi12_unfreeze_last2_001,124486683.0,14222619.0,768.0,gmtt0zej
8,BPI12,gpt2,"-1,-2",0.00500,64.0,['activity'],['accumulated_time'],25,0.8074,0.0102,0.7850,0.8193,extra_feature_bpi12_unfreeze_last2_001,124485915.0,14221851.0,768.0,irtj3frs
9,BPI12,gpt2,"-1,-2",0.00500,64.0,['activity'],[],5,0.8058,0.0022,0.8035,0.8089,extra_features_bpi12_unfreeze_last2_001,124484379.0,14220315.0,768.0,d043j448
